# Đối chứng hư-từ ở n=500 - bịt chỗ hở cuối

Khẳng định *"hư hại ngữ pháp là chuyện **granularity của bộ nén**, không phải
**loại hình ngôn ngữ**"* dựa trên đối chứng tiếng Anh: ở mức nén khớp, MuSiQue
chênh **+0.39** còn VimQA **+0.31** - tức tiếng Anh chênh *nhiều hơn*, ngược
dự đoán loại hình học.

Cả hai con số đo ở **n=50**. Đó là khẳng định load-bearing duy nhất còn ở cỡ mẫu
đó; mọi cái khác đã lên n≥200.

**Kernel CPU** - LLMLingua-2 là BERT-base 700MB, không cần GPU, **không tốn
quota GPU**. Đo lại ở n=500 cho cả hai dataset.

Nếu dấu giữ nguyên (en > vi) → khẳng định vững ở cỡ mẫu gấp mười.
Nếu đảo → phải rút lại, và đó là điều phải biết trước khi nộp.

In [ ]:
!pip install -q llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -2


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats', 'budget']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'✓ tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean, budget_filter
print('cwd:', os.getcwd(), '| scripts/:', os.path.isdir('scripts'))
print(f'✓ đủ {len(NEED)} module')

In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# ══ Quet muc nen + do ti so hu-tu, n=500, CPU ══
import subprocess, sys, os, time
os.makedirs('results', exist_ok=True)
def run_stream(cmd, logfile):
    t0=time.time()
    with open(logfile,'w') as lf:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}'); return p.returncode

for ds in ('vimqa','musique'):
    OUT=f'results/damage_curve_{ds}_500.json'
    if os.path.exists(OUT):
        print(f'[bo qua] da co {OUT}'); continue
    rc=run_stream([sys.executable,'-u','scripts/damage_curve.py',
        '--dataset',ds,'--limit','500','--scorer','bm25','--out',OUT],
        f'results/log_damage_{ds}_500.txt')
    assert rc==0, f'{ds} that bai (rc={rc})'

In [ ]:
# Dau co giu nguyen o n=500?
import json
print(f"{'dataset':10s} {'IterCOMP':>9s} {'LLMLingua-2':>12s} {'chenh':>7s}")
out={}
for ds,lab in (('vimqa','VimQA (vi)'),('musique','MuSiQue (en)')):
    try: d=json.load(open(f'results/damage_curve_{ds}_500.json'))
    except FileNotFoundError: print(f'{lab:10s} (chua co)'); continue
    print(f'  {lab}: xem file, khoa =', list(d.keys())[:6])
    out[ds]=d
print('\nn=50 da co: VimQA +0.31, MuSiQue +0.39 (tieng Anh chenh NHIEU HON)')

In [ ]:
import shutil, os
BASE='/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z=shutil.make_archive(os.path.join(BASE,'results_damage500'),'zip','results')
print('✓',z,f'({os.path.getsize(z)/1e6:.1f} MB)')